In [ ]:
#load data
import pandas as pd

In [ ]:
#show columns in df
df = pd.read_parquet("ts_players.parquet")

print(df.columns)
pd.set_option('display.max_columns', None)
print(df.head())

In [ ]:
#splitting data
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]



In [ ]:
#baseline model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

#run the test
y_pred = model.predict(X_test)

from sklearn.metrics import mean_absolute_error
#calculate MAE
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)


import matplotlib.pyplot as plt
#plot actual vs predicted
plt.figure()
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("Actual vs Predicted Players")
plt.show()

In [ ]:
#advanced model
#lets try XGboost 

from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=True
)
#predict
y_pred = model.predict(X_test)

#evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

import matplotlib.pyplot as plt

plt.figure()
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("XGBoost: Actual vs Predicted Players")
plt.show()

import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

In [ ]:
#different variations of the model